In [ ]:
# source .venv/bin/activate

In [2]:
from dotenv import load_dotenv
import os
from googleapiclient.discovery import build

load_dotenv()

API_KEY = os.getenv("YOUTUBE_API_KEY")
youtube = build("youtube", "v3", developerKey=API_KEY)

print("Connected to YouTube API")

Connected to YouTube API


In [3]:
import sys
print(sys.version)
print(sys.executable)

3.13.7 (main, Aug 14 2025, 11:12:11) [Clang 17.0.0 (clang-1700.0.13.3)]
/Users/nipuninavodyakahandawa/Documents/youtube_pipeline/.venv/bin/python


In [5]:
CHANNEL_ID = "UCRDDfbYPHX_GUJ4lcQYTc8A"

resp = youtube.channels().list(
    part="snippet,contentDetails",
    id=CHANNEL_ID
).execute()

channel_title = resp["items"][0]["snippet"]["title"]
uploads_playlist_id = resp["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

channel_title, uploads_playlist_id

('TV Derana', 'UURDDfbYPHX_GUJ4lcQYTc8A')

In [6]:
resp = youtube.channels().list(
    part="snippet",
    forHandle="tvderana"
).execute()

resp["items"][0]["id"]

'UCc9qiiRqqKKo9rgyj3bysAg'

In [7]:
pl = youtube.playlistItems().list(
    part="contentDetails",
    playlistId=uploads_playlist_id,
    maxResults=5
).execute()

[(item["contentDetails"]["videoId"],
  item["contentDetails"]["videoPublishedAt"])
 for item in pl["items"]]

[('CmDO45ny1oY', '2026-02-25T14:30:06Z'),
 ('o1PUdrlvrSo', '2026-02-25T13:30:06Z'),
 ('TM3LNLZ0XWs', '2026-02-25T13:00:06Z'),
 ('7cTM9EEnQn4', '2026-02-25T12:34:49Z'),
 ('DbB8HBsA3_0', '2026-02-25T12:14:29Z')]

In [9]:
video_id = "CmDO45ny1oY"

v = youtube.videos().list(
    part="snippet,contentDetails,statistics",
    id=video_id
).execute()

item = v["items"][0]
item.keys()

dict_keys(['kind', 'etag', 'id', 'snippet', 'contentDetails', 'statistics'])

In [12]:
import isodate
from datetime import datetime, timezone

item = v["items"][0]

# Convert duration ISO8601 (e.g., PT5M12S) to seconds
duration_seconds = int(
    isodate.parse_duration(item["contentDetails"]["duration"]).total_seconds()
)

# Capture fields (UTC)
now_utc = datetime.now(timezone.utc)

extracted = {
    # Required fields
    "video_id": item["id"],
    "video_title": item["snippet"]["title"],
    "channel_name": item["snippet"]["channelTitle"],
    "playlist_id": uploads_playlist_id,           # Using Uploads playlist
    "playlist_name": "Uploads",
    "channel_id": item["snippet"]["channelId"],
    "publish_datetime": item["snippet"]["publishedAt"],
    "video_duration": duration_seconds,           # Duration in seconds
    "like_count": int(item["statistics"].get("likeCount", 0)),
    "view_count": int(item["statistics"].get("viewCount", 0)),
    "comment_count": int(item["statistics"].get("commentCount", 0)),
    "data_capture_date": now_utc.date().isoformat(),
    "data_capture_timestamp_utc": now_utc.isoformat(),
}

extracted

{'video_id': 'CmDO45ny1oY',
 'video_title': 'Iskole (ඉස්කෝලේ) | Episode 1293 | 25th February 2026',
 'channel_name': 'TV Derana',
 'playlist_id': 'UURDDfbYPHX_GUJ4lcQYTc8A',
 'playlist_name': 'Uploads',
 'channel_id': 'UCRDDfbYPHX_GUJ4lcQYTc8A',
 'publish_datetime': '2026-02-25T14:30:06Z',
 'video_duration': 1124,
 'like_count': 583,
 'view_count': 19099,
 'comment_count': 18,
 'data_capture_date': '2026-02-25',
 'data_capture_timestamp_utc': '2026-02-25T16:25:37.550715+00:00'}

In [11]:
import isodate

dur_iso = extracted["duration_iso8601"]
dur_seconds = int(isodate.parse_duration(dur_iso).total_seconds())
dur_iso, dur_seconds

('PT18M44S', 1124)

In [13]:
video_ids = [it["contentDetails"]["videoId"] for it in pl["items"]]
video_ids

['CmDO45ny1oY', 'o1PUdrlvrSo', 'TM3LNLZ0XWs', '7cTM9EEnQn4', 'DbB8HBsA3_0']

In [14]:
v = youtube.videos().list(
    part="snippet,contentDetails,statistics",
    id=",".join(video_ids)
).execute()

len(v["items"])


5

In [16]:
import isodate
from datetime import datetime, timezone

now_utc = datetime.now(timezone.utc)

rows = []

for item in v["items"]:
    
    # Safe duration handling
    duration_iso = item.get("contentDetails", {}).get("duration")
    
    if duration_iso:
        duration_seconds = int(isodate.parse_duration(duration_iso).total_seconds())
    else:
        duration_seconds = 0
    
    row = {
        "video_id": item["id"],
        "video_title": item["snippet"]["title"],
        "channel_name": item["snippet"]["channelTitle"],
        "playlist_id": uploads_playlist_id,
        "playlist_name": "Uploads",
        "channel_id": item["snippet"]["channelId"],
        "publish_datetime": item["snippet"]["publishedAt"],
        "video_duration": duration_seconds,
        "like_count": int(item.get("statistics", {}).get("likeCount", 0)),
        "view_count": int(item.get("statistics", {}).get("viewCount", 0)),
        "comment_count": int(item.get("statistics", {}).get("commentCount", 0)),
        "data_capture_date": now_utc.date().isoformat(),
        "data_capture_timestamp_utc": now_utc.isoformat(),
    }
    
    rows.append(row)

rows[:2], len(rows)

([{'video_id': 'CmDO45ny1oY',
   'video_title': 'Iskole (ඉස්කෝලේ) | Episode 1293 | 25th February 2026',
   'channel_name': 'TV Derana',
   'playlist_id': 'UURDDfbYPHX_GUJ4lcQYTc8A',
   'playlist_name': 'Uploads',
   'channel_id': 'UCRDDfbYPHX_GUJ4lcQYTc8A',
   'publish_datetime': '2026-02-25T14:30:06Z',
   'video_duration': 1124,
   'like_count': 671,
   'view_count': 19872,
   'comment_count': 20,
   'data_capture_date': '2026-02-25',
   'data_capture_timestamp_utc': '2026-02-25T16:34:20.700006+00:00'},
  {'video_id': 'o1PUdrlvrSo',
   'video_title': 'Sitha Nidi Na (සිත නිදි නෑ) | Episode 626 | 25th February 2026',
   'channel_name': 'TV Derana',
   'playlist_id': 'UURDDfbYPHX_GUJ4lcQYTc8A',
   'playlist_name': 'Uploads',
   'channel_id': 'UCRDDfbYPHX_GUJ4lcQYTc8A',
   'publish_datetime': '2026-02-25T13:30:06Z',
   'video_duration': 1133,
   'like_count': 218,
   'view_count': 6064,
   'comment_count': 6,
   'data_capture_date': '2026-02-25',
   'data_capture_timestamp_utc': '2026-02-